In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import preprocessing
from sklearn import metrics
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score

In [2]:
from datetime import timedelta

In [3]:
## Loading the data file

hr_data=pd.read_csv('../data/HR_pre_processed_data_2.csv').iloc[0:1470,1:]
hr_data.head()

,Age,Department,Education,EducationField,Gender,JobLevel,MonthlyIncome,PercentSalaryHike,TotalWorkingYears,YearsAtCompany,Date,ZIP
0,41,2,2,1,0,2,5993,11,8,6,3/2/2007,44039
1,49,1,1,1,1,2,5130,23,10,10,3/22/2007,55710
2,37,1,2,4,1,1,2090,15,7,0,4/6/2007,80550
3,33,1,4,1,0,1,2909,11,8,8,4/14/2007,46777
4,27,1,1,3,1,1,3468,12,6,2,4/15/2007,33109


In [4]:
import pgeocode

In [5]:
# Using PGEOCODE
df=hr_data.copy(deep=True)

nomi = pgeocode.Nominatim('us')

df['county']=nomi.query_postal_code(df['ZIP'].values).county_name
df['state']=nomi.query_postal_code(df['ZIP'].values).state_name
df['ZIP3']=df['ZIP']
df['ZIP3']=df['ZIP3'].apply(lambda x: str(x)[:3]+'XX' )

df.head()

,Age,Department,Education,EducationField,Gender,JobLevel,MonthlyIncome,PercentSalaryHike,TotalWorkingYears,YearsAtCompany,Date,ZIP,county,state,ZIP3
0,41,2,2,1,0,2,5993,11,8,6,3/2/2007,44039,Lorain,Ohio,440XX
1,49,1,1,1,1,2,5130,23,10,10,3/22/2007,55710,St. Louis,Minnesota,557XX
2,37,1,2,4,1,1,2090,15,7,0,4/6/2007,80550,Weld,Colorado,805XX
3,33,1,4,1,0,1,2909,11,8,8,4/14/2007,46777,Wells,Indiana,467XX
4,27,1,1,3,1,1,3468,12,6,2,4/15/2007,33109,Miami-Dade,Florida,331XX


In [6]:
import hashlib
#Hashing any string / categroical field
feature=['Gender','JobLevel']
#print(df["Gender"].unique())
for f in feature:
    df[f] = df[f].apply(lambda x: hashlib.sha1(repr(x).encode('utf-8')).hexdigest())    

df

,Age,Department,Education,EducationField,Gender,JobLevel,MonthlyIncome,PercentSalaryHike,TotalWorkingYears,YearsAtCompany,Date,ZIP,county,state,ZIP3
0,41,2,2,1,b6589fc6ab0dc82cf12099d1c2d40ab994e8410c,da4b9237bacccdf19c0760cab7aec4a8359010b0,5993,11,8,6,3/2/2007,44039,Lorain,Ohio,440XX
1,49,1,1,1,356a192b7913b04c54574d18c28d46e6395428ab,da4b9237bacccdf19c0760cab7aec4a8359010b0,5130,23,10,10,3/22/2007,55710,St. Louis,Minnesota,557XX
2,37,1,2,4,356a192b7913b04c54574d18c28d46e6395428ab,356a192b7913b04c54574d18c28d46e6395428ab,2090,15,7,0,4/6/2007,80550,Weld,Colorado,805XX
3,33,1,4,1,b6589fc6ab0dc82cf12099d1c2d40ab994e8410c,356a192b7913b04c54574d18c28d46e6395428ab,2909,11,8,8,4/14/2007,46777,Wells,Indiana,467XX
4,27,1,1,3,356a192b7913b04c54574d18c28d46e6395428ab,356a192b7913b04c54574d18c28d46e6395428ab,3468,12,6,2,4/15/2007,33109,Miami-Dade,Florida,331XX
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1465,36,1,2,3,356a192b7913b04c54574d18c28d46e6395428ab,da4b9237bacccdf19c0760cab7aec4a8359010b0,2571,17,17,5,6/16/2015,18461,Wayne,Pennsylvania,184XX
1466,39,1,1,3,356a192b7913b04c54574d18c28d46e6395428ab,77de68daecd823babbb58edb1c8e14d7106e83bb,9991,15,9,7,6/16/2015,84026,Uintah,Utah,840XX
1467,27,1,3,1,356a192b7913b04c54574d18c28d46e6395428ab,da4b9237bacccdf19c0760cab7aec4a8359010b0,6142,20,6,6,6/16/2015,68964,Nuckolls,Nebraska,689XX
1468,49,2,3,3,356a192b7913b04c54574d18c28d46e6395428ab,da4b9237bacccdf19c0760cab7aec4a8359010b0,5390,14,17,9,6/15/2015,59231,Valley,Montana,592XX
